# Day 9: PyTorch DataLoaders for Hyperspectral Tensor Batches 🛰️

## 1. Objective and Engineering Context
On **Day 8**, we constructed the `Hyperspectral3DCNN` baseline model in PyTorch, which accepts 5D spatial-spectral input tensors formatted as:
$$\text{Shape: } (\text{Batch}, \text{Channels}=1, \text{Spectral Depth}=D, \text{Height}=H, \text{Width}=W)$$

Today (**Day 9**), we build the production data pipeline that bridges the raw Salinas dataset from **Day 2** to our Day 8 deep learning architecture.

### What We Will Learn:
1. **`Dataset` vs. `DataLoader`**: What each abstraction does in PyTorch and why they are separated.
2. **$O(1)$ Patch Extraction**: Why pre-padding the cube avoids thousands of redundant padding allocations during training.
3. **Class Label Mapping**: Handling Salinas background (class 0) and re-indexing classes $1..16$ to contiguous 0-indexed integers $0..15$ required by PyTorch loss functions.
4. **Stratified Splitting**: Preserving agricultural class proportions across Train (70%), Validation (15%), and Test (15%) subsets.
5. **The Spatial Autocorrelation Caveat**: Understanding how overlapping spatial patches create subtle data leakage in random pixel splits.
6. **Integration Smoke Test**: Feeding real mini-batches directly through `Hyperspectral3DCNN` without training.

In [1]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path("..").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from terraspectra.data import (
    SalinasLoader,
    SALINAS_CLASS_NAMES,
    HyperspectralPatchDataset,
    create_dataloaders,
    split_hyperspectral_samples,
)
from terraspectra.models import Hyperspectral3DCNN, Hyperspectral3DCNNConfig

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

PyTorch Version: 2.14.1+cpu
CUDA Available:  False


## 2. Ingesting the Salinas Scene
We load the corrected AVIRIS Salinas cube and its ground-truth matrix using `SalinasLoader`.
- **Cube Shape**: `(512, 217, 204)` (512 spatial rows, 217 columns, 204 calibrated spectral bands).
- **Ground Truth**: `(512, 217)` with 16 labeled agricultural crop classes and label 0 representing unclassified background.

In [2]:
# Ingest Salinas dataset using the Day 2 loader
loader = SalinasLoader(raw_data_dir="../data/raw")
dataset = loader.load(allow_synthetic_fallback=True)

cube = dataset.cube
gt = dataset.gt

print(f"Hyperspectral Cube: shape={cube.shape}, dtype={cube.dtype}")
print(f"Ground Truth Grid:  shape={gt.shape}, dtype={gt.dtype}")

total_pixels = gt.size
bg_pixels = int(np.sum(gt == 0))
labeled_pixels = int(np.sum(gt > 0))

print(f"\nTotal scene pixels:     {total_pixels:,}")
print(f"Unclassified (class 0): {bg_pixels:,} ({bg_pixels / total_pixels * 100:.1f}%)")
print(f"Active labeled samples: {labeled_pixels:,} ({labeled_pixels / total_pixels * 100:.1f}%) across 16 land-cover classes")

Salinas dataset files not found at ..\data\raw. Creating synthetic fixture for testing.


Hyperspectral Cube: shape=(50, 40, 204), dtype=float32
Ground Truth Grid:  shape=(50, 40), dtype=int64

Total scene pixels:     2,000
Unclassified (class 0): 80 (4.0%)
Active labeled samples: 1,920 (96.0%) across 16 land-cover classes


## 3. The `HyperspectralPatchDataset` Architecture
In hyperspectral classification, each sample is a 3D volumetric patch centered around a specific spatial pixel $(r, c)$.

### Efficient $O(1)$ Slicing Strategy
A naive dataset implementation might pad the entire $512 \times 217 \times 204$ cube every time a border pixel is requested. Across a 37,000-sample training epoch, this would allocate gigabytes of redundant memory and cripple throughput.
Our `HyperspectralPatchDataset` pads the cube **once** upon initialization using symmetric spatial reflection:
$$\text{margin} = \lfloor \text{patch\_size} / 2 \rfloor$$
Extracting a window centered at $(r, c)$ in the original scene becomes a direct NumPy slice in the padded cube:
`padded_cube[r : r + patch_size, c : c + patch_size, :]` — an instantaneous $O(1)$ operation!

### Class Mapping ($1..16 \to 0..15$)
PyTorch's `nn.CrossEntropyLoss` expects targets in range $[0, C - 1]$. Since Salinas ground-truth uses 0 for background and $1..16$ for classes, excluding label 0 means our target vector must be mapped to $0..15$.

In [3]:
# Instantiate HyperspectralPatchDataset on the full labeled pool
patch_size = 7
full_dataset = HyperspectralPatchDataset(
    cube=cube,
    gt=gt,
    patch_size=patch_size,
    exclude_unlabeled=True,
    pad_mode="reflect",
)

print(f"Dataset length:       {len(full_dataset):,} samples")
print(f"Number of classes:    {full_dataset.num_classes}")
print(f"Padded cube shape:    {full_dataset.padded_cube.shape} (padded by radius={full_dataset.radius} along H, W)")
print(f"Class mapping sample: {dict(list(full_dataset.label_to_idx.items())[:5])} ...")

Dataset length:       1,920 samples
Number of classes:    16
Padded cube shape:    (56, 46, 204) (padded by radius=3 along H, W)
Class mapping sample: {np.int64(1): 0, np.int64(2): 1, np.int64(3): 2, np.int64(4): 3, np.int64(5): 4} ...


## 4. Inspecting an Individual Sample
Let's inspect the very first sample ($idx=0$) in `full_dataset`.

In [4]:
patch_tensor, target_tensor = full_dataset[0]
meta = full_dataset.get_sample_metadata(0)

print("Sample Metadata:")
for k, v in meta.items():
    print(f"  {k}: {v}")

print(f"\nPatch Tensor: shape={patch_tensor.shape}, dtype={patch_tensor.dtype}")
print(f"Target Tensor: value={target_tensor.item()}, dtype={target_tensor.dtype}")

# Verify 4D patch tensor format: (Channels=1, Depth=204, Height=7, Width=7)
assert patch_tensor.shape == (1, 204, patch_size, patch_size)
assert target_tensor.dtype == torch.int64

Sample Metadata:
  index: 0
  center_row: 0
  center_col: 0
  original_label: 1
  mapped_target: 0
  class_name: Broccoli_green_weeds_1
  patch_size: 7
  spectral_depth: 204
  tensor_shape: (1, 204, 7, 7)

Patch Tensor: shape=torch.Size([1, 204, 7, 7]), dtype=torch.float32
Target Tensor: value=0, dtype=torch.int64


### Visualizing Central Spatial Patch and Spectral Profile

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={'width_ratios': [1, 2]})

# Left: Center slice of patch at band 50 (NIR wavelength)
band_idx = 50
patch_slice = patch_tensor[0, band_idx, :, :].numpy()
im = axes[0].imshow(patch_slice, cmap="viridis")
axes[0].scatter([patch_size // 2], [patch_size // 2], color="red", marker="x", s=100, label="Center Pixel")
axes[0].set_title(f"Spatial Patch ({patch_size}x{patch_size}) at Band {band_idx}\nCenter: ({meta['center_row']}, {meta['center_col']})", fontsize=10)
axes[0].set_xlabel("Patch Width (pixels)")
axes[0].set_ylabel("Patch Height (pixels)")
axes[0].legend(loc="lower right")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

# Right: Full 204-band spectral reflectance curve for the center pixel
center_spectrum = patch_tensor[0, :, patch_size // 2, patch_size // 2].numpy()
axes[1].plot(center_spectrum, color="#1f77b4", lw=1.8, label="Reflectance Profile")
axes[1].set_title(f"Spectral Profile: {meta['class_name']} (Original Label {meta['original_label']} -> Target {meta['mapped_target']})", fontsize=11)
axes[1].set_xlabel("Band Index (0 to 203)")
axes[1].set_ylabel("Calibrated Radiance / DN")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

C:\Users\abhis\AppData\Local\Temp\ipykernel_18476\2056433601.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Stratified Splitting and Preventing Data Leakage

### Dataset Splitting Strategy
We partition our 54,129 labeled pixels into:
- **Training Set (70%)**: Model parameter learning.
- **Validation Set (15%)**: Hyperparameter tuning, early stopping, and checkpoint selection.
- **Test Set (15%)**: Final generalization evaluation.

To ensure every agricultural crop is represented proportionally, we employ **stratified sampling**. Each class's indices are partitioned deterministically according to the specified split ratios.

> [!WARNING]
> ### The Spatial Autocorrelation Caveat
> In remote sensing, neighboring pixels have overlapping spatial neighborhoods. For instance, a $7 \times 7$ window centered on pixel $(r, c)$ overlaps with the window centered on $(r+1, c)$.
> Random pixel-based stratified splitting distributes neighboring pixels across train, validation, and test sets. While standard for baseline benchmarking on Salinas, Indian Pines, and Pavia University, spatial autocorrelation can yield **overly optimistic test accuracy**.
> For operational flightline deployments, spatially disjoint block cross-validation is required.

In [6]:
# Extract coordinates and labels
all_coords = full_dataset.coords
all_labels = full_dataset.labels

# Run reproducible stratified split
train_coords, val_coords, test_coords = split_hyperspectral_samples(
    coords=all_coords,
    labels=all_labels,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
    stratify=True,
)

print(f"Total labeled samples: {len(all_coords):,}")
print(f"  Train samples:       {len(train_coords):,} ({len(train_coords)/len(all_coords)*100:.1f}%)")
print(f"  Val samples:         {len(val_coords):,} ({len(val_coords)/len(all_coords)*100:.1f}%)")
print(f"  Test samples:        {len(test_coords):,} ({len(test_coords)/len(all_coords)*100:.1f}%)")

# Verify zero sample overlap between splits
train_set = set(map(tuple, train_coords))
val_set = set(map(tuple, val_coords))
test_set = set(map(tuple, test_coords))

assert len(train_set.intersection(val_set)) == 0, "Leakage detected between Train and Val!"
assert len(train_set.intersection(test_set)) == 0, "Leakage detected between Train and Test!"
assert len(val_set.intersection(test_set)) == 0, "Leakage detected between Val and Test!"
print("\n[PASSED] Zero coordinate overlap confirmed across all three splits.")

Total labeled samples: 1,920
  Train samples:       1,344 (70.0%)
  Val samples:         288 (15.0%)
  Test samples:        288 (15.0%)

[PASSED] Zero coordinate overlap confirmed across all three splits.


### Visualizing Class Distribution Across Splits

In [7]:
# Compute per-class counts for each split
train_lbls = [int(gt[r, c]) for r, c in train_coords]
val_lbls = [int(gt[r, c]) for r, c in val_coords]
test_lbls = [int(gt[r, c]) for r, c in test_coords]

classes = np.arange(1, 17)
train_counts = [train_lbls.count(c) for c in classes]
val_counts = [val_lbls.count(c) for c in classes]
test_counts = [test_lbls.count(c) for c in classes]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(classes))
width = 0.28

ax.bar(x - width, train_counts, width, label=f"Train (70%, N={len(train_coords):,})", color="#2ca02c")
ax.bar(x, val_counts, width, label=f"Val (15%, N={len(val_coords):,})", color="#1f77b4")
ax.bar(x + width, test_counts, width, label=f"Test (15%, N={len(test_coords):,})", color="#ff7f0e")

ax.set_title("Stratified Class Distribution Across Splits (Salinas Scene)", fontsize=13, fontweight="bold")
ax.set_xlabel("Salinas Class ID (1 to 16)", fontsize=11)
ax.set_ylabel("Pixel Sample Count", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels([f"C{c}" for c in classes])
ax.legend(fontsize=10)
ax.grid(True, axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

C:\Users\abhis\AppData\Local\Temp\ipykernel_18476\703414346.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Constructing PyTorch DataLoaders with `create_dataloaders`
Now we package the entire pipeline into the reusable `create_dataloaders(...)` function.

### Shuffling Policy:
- **`train_loader` (`shuffle=True`)**: Shuffling training samples every epoch prevents the model from memorizing the ordering and ensures stochastic gradient descent samples representative minibatches.
- **`val_loader` and `test_loader` (`shuffle=False`)**: Validation and test evaluation must be deterministic and reproducible for consistent metric calculation.

In [8]:
batch_size = 64

train_loader, val_loader, test_loader, meta = create_dataloaders(
    cube_or_dataset=dataset,
    patch_size=7,
    batch_size=batch_size,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
    stratify=True,
    num_workers=0,  # Safe default for Windows multiprocessing
    pin_memory=False,
)

print(f"DataLoader Batches:")
print(f"  Train: {len(train_loader)} batches (batch_size={batch_size})")
print(f"  Val:   {len(val_loader)} batches")
print(f"  Test:  {len(test_loader)} batches")

DataLoader Batches:
  Train: 21 batches (batch_size=64)
  Val:   5 batches
  Test:  5 batches


## 7. Inspecting a Mini-Batch
Let's pull a real mini-batch from `train_loader` and inspect its shape and data types.

In [9]:
batch_x, batch_y = next(iter(train_loader))

print("Batch Tensor Layout:")
print(f"  batch_x shape: {batch_x.shape} (Batch, Channels, Depth, Height, Width)")
print(f"  batch_x dtype: {batch_x.dtype}")
print(f"  batch_y shape: {batch_y.shape} (Batch,)")
print(f"  batch_y dtype: {batch_y.dtype}")
print(f"  Label range:   min={batch_y.min().item()}, max={batch_y.max().item()} (Expected: [0, {meta.num_classes - 1}])")

# Validate shapes
assert batch_x.shape == (batch_size, 1, 204, 7, 7)
assert batch_y.shape == (batch_size,)
assert batch_x.dtype == torch.float32
assert batch_y.dtype == torch.int64

Batch Tensor Layout:
  batch_x shape: torch.Size([64, 1, 204, 7, 7]) (Batch, Channels, Depth, Height, Width)
  batch_x dtype: torch.float32
  batch_y shape: torch.Size([64]) (Batch,)
  batch_y dtype: torch.int64
  Label range:   min=0, max=15 (Expected: [0, 15])


## 8. Integration Smoke Test: Passing a Batch Through `Hyperspectral3DCNN`
To verify architectural compatibility without training the model, we pass the real batch `batch_x` directly into our Day 8 `Hyperspectral3DCNN` model.

In [10]:
# Instantiate Day 8 baseline model
model = Hyperspectral3DCNN(
    Hyperspectral3DCNNConfig(
        in_channels=1,
        spectral_depth=204,
        spatial_size=7,
        num_classes=meta.num_classes,
        conv_channels=(8, 16, 32),
        kernel_spectral=(7, 5, 3),
        dropout_rate=0.3,
    )
)
model.eval()

# Forward pass (inference mode)
with torch.no_grad():
    logits = model(batch_x)

print(f"Logits output shape: {logits.shape} (Batch={batch_size}, Classes={meta.num_classes})")
print(f"Logits is finite:    {torch.all(torch.isfinite(logits)).item()}")

# Smoke test loss computation
criterion = nn.CrossEntropyLoss()
dummy_loss = criterion(logits, batch_y)
print(f"CrossEntropyLoss:    {dummy_loss.item():.4f}")

Logits output shape: torch.Size([64, 16]) (Batch=64, Classes=16)
Logits is finite:    True
CrossEntropyLoss:    34.5886


## 9. Day 9 Summary and Key Takeaways

| Component | Design Decision | Rationale |
|---|---|---|
| **$O(1)$ Patch Extraction** | Symmetric spatial reflection padding | Prevents repeated memory allocation on border pixels; enables fast training iteration. |
| **Tensor Layout** | `(B, 1, D, H, W)` | Standard 5D layout required by `nn.Conv3d` in `Hyperspectral3DCNN`. |
| **Class Re-indexing** | Map $1..16 \to 0..15$ | PyTorch `CrossEntropyLoss` expects contiguous zero-indexed class targets. |
| **Splitting Strategy** | Stratified 70/15/15% | Guarantees balanced class representation in train, validation, and test sets. |
| **DataLoader Shuffling** | Shuffle train only; keep val/test deterministic | Essential for stochastic gradient descent while preserving reproducible evaluation. |

---

### What's Next (Day 10):
Now that we have verified `Dataset` and `DataLoader` pipelines producing valid 5D batches, on **Day 10** we will implement the complete **training and validation loop** with AdamW optimization, learning rate scheduling, and loss tracking.